# Importing Libraries

In [1]:
# Load the Drive helper and mount
#from google.colab import drive
#drive.mount('/content/drive')

In [44]:
#!pip install torch_geometric_temporal
#!pip install networkx

import os
import torch
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.models import load_model

In [3]:
# Set display option to show all rows 
pd.set_option('display.max_rows', None)

# Set display option to show all columns 
pd.set_option('display.max_columns', None)

# Importing Dataset

In [4]:
#path = "/content/drive/MyDrive/StationData"
path = ".."

node_df = pd.read_csv(path + "/StationData/StationsData.csv", sep=";", skiprows=[0], header=None,
                 names=["ComputerNo", "StationName", "Latitude", "Longitude", "Altitude"])

In [5]:
data1_df = pd.read_csv(path + "/StationData/St_BrandvleiB.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data2_df = pd.read_csv(path + "/StationData/St_Kakamas.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data3_df = pd.read_csv(path + "/StationData/St_Luckhof.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data4_df = pd.read_csv(path + "/StationData/St_Olifantshoek.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data5_df = pd.read_csv(path + "/StationData/St_Prieska.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data6_df = pd.read_csv(path + "/StationData/St_Vaalharts.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data7_df = pd.read_csv(path + "/StationData/St_VanWyksvleiB.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

data8_df = pd.read_csv(path + "/StationData/St_BerkleyWest.csv", sep=";", skiprows=[0], header=None,
                 names=["Compno", "Year", "Month", "AverageMaximumTemperature", "AverageMinimunTemperature", "AverageMaximumRelativeHumidity", "AverageMinimumRelativeHumidity", "TotalRainfall", "AverageTotalRadiation", "AverageWindSpeed", "AverageTotalRelativeEvaporation"])

frames = [data1_df, data2_df, data3_df, data4_df, data5_df, data6_df, data7_df, data8_df]
attribute_df = pd.concat(frames)

# Data Cleaning

## Remove commas and nulls

In [6]:
attribute_df = attribute_df.replace(',', '.', regex=True)
attribute_df = attribute_df.replace('~', '0', regex=True)

node_df = node_df.replace(',', '.', regex=True)
node_df = node_df.replace('~', '0', regex=True)

## Convert to numeric

In [7]:
attribute_df = attribute_df.apply(pd.to_numeric)

node_df[['ComputerNo', 'Latitude', 'Longitude', 'Altitude']] = node_df[['ComputerNo', 'Latitude', 'Longitude', 'Altitude']].apply(pd.to_numeric)

## Fixing Null Values

In [8]:
attribute_df['TotalRainfall'] = attribute_df['TotalRainfall'].fillna(method='ffill')

In [9]:
node_df = node_df.rename(columns={'ComputerNo': 'Compno'})

In [10]:
attribute_df.head()

,Compno,Year,Month,AverageMaximumTemperature,AverageMinimunTemperature,AverageMaximumRelativeHumidity,AverageMinimumRelativeHumidity,TotalRainfall,AverageTotalRadiation,AverageWindSpeed,AverageTotalRelativeEvaporation
0,30692,2004,11,32.78,14.63,62.34,13.02,2.4,28.86,3.50,134.03
1,30692,2004,12,34.16,15.41,65.96,11.75,4.0,30.99,3.77,242.41
2,30692,2005,1,34.59,16.57,68.01,13.89,19.2,28.71,3.68,230.03
3,30692,2005,2,35.79,17.49,63.55,14.27,16.4,26.59,3.02,191.89
4,30692,2005,3,31.21,14.96,74.17,19.67,14.8,21.49,2.73,160.74


In [11]:
node_df.head()

,Compno,StationName,Latitude,Longitude,Altitude
0,30688,VANWYKSVLEI,-30.40826,21.855016,999
1,30142,JANKEMPDORP; VAALHARTS,-27.95770,24.839940,1165
2,30173,KAKAMAS; KROMHOUT BOERDERY,-28.78690,20.655190,662
3,30191,PRIESKA; BLAAUKRANS,-29.52531,22.972690,943
4,30389,LUCKHOF; BLESKOP,-29.87956,24.582830,1119


In [12]:
graph_df = pd.merge(attribute_df, node_df, on='Compno', how='inner')

In [13]:
graph_df.drop(['StationName'], axis=1, inplace=True)

In [14]:
graph_df.insert(1, 'TimeStamp', graph_df.apply(lambda row: str(row['Year']).replace('.0', '') + "-" + str(row['Month']).replace('.0', ''), axis=1))

In [15]:
# Convert to Date format
graph_df['TimeStamp'] = pd.to_datetime(graph_df['TimeStamp'])

In [16]:
# Sorting dataset by TimeStamp
graph_df = graph_df.sort_values(by='TimeStamp').reset_index()

In [17]:
graph_df = graph_df[['index', 'TimeStamp', 'Compno', 'Latitude', 'Longitude', 'Altitude', 'Year', 'Month', 'AverageMaximumTemperature', 'AverageMinimunTemperature', 'AverageMaximumRelativeHumidity', 'AverageMinimumRelativeHumidity', 'TotalRainfall', 'AverageTotalRadiation', 'AverageWindSpeed', 'AverageTotalRelativeEvaporation']]
graph_df.drop(['index', 'Year', 'Month'], axis=1, inplace=True)

## Creating Target

In [18]:
# Specify columns to normalize
columns_to_normalize = ['Latitude', 'Longitude', 'Altitude', 'AverageMaximumTemperature', 'AverageMinimunTemperature', 'AverageMaximumRelativeHumidity', 'AverageMinimumRelativeHumidity', 'TotalRainfall', 'AverageTotalRadiation', 'AverageWindSpeed', 'AverageTotalRelativeEvaporation']

# Initialize the scaler
scaler = MinMaxScaler()

# Apply the scaler to the specified columns
graph_df[columns_to_normalize] = scaler.fit_transform(graph_df[columns_to_normalize])

In [19]:
graph_df['Target'] = graph_df.drop(['Compno', 'TimeStamp'], axis=1).apply(lambda x: x.mean(), axis=1)

In [20]:
# Create 5 quantile-based categories
graph_df['Quantile_Category'] = pd.qcut(graph_df['Target'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

## Data Encoding

In [21]:
# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Encode the 'Category' column
graph_df['Category_encoded'] = label_encoder.fit_transform(graph_df['Quantile_Category'])

In [22]:
columns_list = graph_df.columns.tolist()
print(columns_list)

['TimeStamp', 'Compno', 'Latitude', 'Longitude', 'Altitude', 'AverageMaximumTemperature', 'AverageMinimunTemperature', 'AverageMaximumRelativeHumidity', 'AverageMinimumRelativeHumidity', 'TotalRainfall', 'AverageTotalRadiation', 'AverageWindSpeed', 'AverageTotalRelativeEvaporation', 'Target', 'Quantile_Category', 'Category_encoded']


In [23]:
final_df = graph_df[['TimeStamp', 'Compno', 'Latitude', 'Longitude', 'Altitude', 'AverageMaximumTemperature', 'AverageMinimunTemperature', 'AverageMaximumRelativeHumidity', 'AverageMinimumRelativeHumidity', 'TotalRainfall', 'AverageTotalRadiation', 'AverageWindSpeed', 'AverageTotalRelativeEvaporation', 'Category_encoded']]

In [24]:
final_df = final_df.rename(columns={'Category_encoded': 'Target'})
time_df = final_df.copy()

In [25]:
# Sort the data by Compno and TimeStamp
final_df = final_df.sort_values(by=['Compno', 'TimeStamp'])

In [26]:
time_df['TimeStamp'] = pd.to_datetime(time_df['TimeStamp'])

In [27]:
# Shift the TimeStamp column by one month
time_df['Shifted_TimeStamp'] = time_df['TimeStamp'] + pd.DateOffset(months=1)

In [28]:
# Sort the data by Compno and TimeStamp to ensure proper alignment
time_corrected_df = time_df.sort_values(by=['Compno', 'TimeStamp']).reset_index()

In [29]:
# Create Shifted Target (NextMonth_Target)
time_corrected_df['Shifted_Target'] = time_corrected_df.apply(
    lambda row: time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] + pd.DateOffset(months=1)), 'Target'
    ].values[0] if not time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] + pd.DateOffset(months=1))
    ].empty else None,
    axis=1
)

# Create Lag1_Target
time_corrected_df['Lag1_Target'] = time_corrected_df.apply(
    lambda row: time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] - pd.DateOffset(months=1)), 'Target'
    ].values[0] if not time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] - pd.DateOffset(months=1))
    ].empty else None,
    axis=1
)

# Create Lag2_Target
time_corrected_df['Lag2_Target'] = time_corrected_df.apply(
    lambda row: time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] - pd.DateOffset(months=2)), 'Target'
    ].values[0] if not time_corrected_df.loc[
        (time_corrected_df['Compno'] == row['Compno']) & 
        (time_corrected_df['TimeStamp'] == row['TimeStamp'] - pd.DateOffset(months=2))
    ].empty else None,
    axis=1
)

In [30]:
analysis_df = time_corrected_df[['TimeStamp', 'Compno', 'Latitude', 'Longitude', 'Altitude', 'AverageMaximumTemperature', 'AverageMinimunTemperature', 'AverageMaximumRelativeHumidity', 'AverageMinimumRelativeHumidity', 'TotalRainfall', 'AverageTotalRadiation', 'AverageWindSpeed', 'AverageTotalRelativeEvaporation', 'Lag1_Target', 'Lag2_Target', 'Shifted_Target']]

In [31]:
analysis_df.describe()

,Compno,Latitude,Longitude,Altitude,AverageMaximumTemperature,AverageMinimunTemperature,AverageMaximumRelativeHumidity,AverageMinimumRelativeHumidity,TotalRainfall,AverageTotalRadiation,AverageWindSpeed,AverageTotalRelativeEvaporation,Lag1_Target,Lag2_Target,Shifted_Target
count,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1302.000000,1282.000000,1264.000000,1282.00000
mean,30427.989247,0.418409,0.491893,0.592993,0.157293,0.057175,0.709463,0.356838,0.077342,0.577404,0.067627,0.523655,2.000780,2.001582,2.00078
std,266.892808,0.379418,0.368395,0.304402,0.043444,0.071970,0.123967,0.139195,0.111215,0.193440,0.044458,0.216832,1.415041,1.419243,1.41228
min,30142.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,30173.000000,0.014587,0.033563,0.476786,0.129454,0.043509,0.629690,0.247652,0.003579,0.421525,0.048989,0.348097,1.000000,1.000000,1.00000
50%,30389.000000,0.353444,0.543639,0.601786,0.159680,0.049393,0.725273,0.337043,0.032928,0.586350,0.063035,0.508653,2.000000,2.000000,2.00000
75%,30688.000000,0.649029,0.940622,0.816071,0.182005,0.054246,0.805055,0.436522,0.107015,0.733559,0.080507,0.693258,3.000000,3.000000,3.00000
max,30899.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000,4.000000,4.00000


In [32]:
# Drop rows where the 'Name' column is null
analysis_df = analysis_df.dropna(subset=['Lag1_Target', 'Lag2_Target', 'Shifted_Target'])
analysis_df = analysis_df.rename(columns={'Shifted_Target': 'Target'})

In [33]:
# Check if there are any null values in the DataFrame
if analysis_df.isnull().values.any():
    print("There are null values in the DataFrame.")
else:
    print("No null values found in the DataFrame.")

No null values found in the DataFrame.


In [34]:
analysis_df.describe()

,Compno,Latitude,Longitude,Altitude,AverageMaximumTemperature,AverageMinimunTemperature,AverageMaximumRelativeHumidity,AverageMinimumRelativeHumidity,TotalRainfall,AverageTotalRadiation,AverageWindSpeed,AverageTotalRelativeEvaporation,Lag1_Target,Lag2_Target,Target
count,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000,1244.000000
mean,30427.345659,0.416610,0.490715,0.590975,0.157069,0.056981,0.711828,0.358420,0.077727,0.574186,0.067467,0.523844,1.997588,2.006431,1.998392
std,266.002688,0.377373,0.369771,0.305442,0.043560,0.072711,0.121795,0.138593,0.111678,0.193126,0.045177,0.212874,1.417337,1.421009,1.418756
min,30142.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30173.000000,0.014587,0.033563,0.476786,0.129399,0.043479,0.630783,0.250913,0.003579,0.420213,0.048989,0.348989,1.000000,1.000000,1.000000
50%,30389.000000,0.353444,0.543639,0.601786,0.158908,0.049251,0.729007,0.338174,0.032928,0.583034,0.062864,0.508135,2.000000,2.000000,2.000000
75%,30688.000000,0.649029,0.940622,0.816071,0.181467,0.054136,0.805852,0.437087,0.108089,0.728792,0.079908,0.691234,3.000000,3.000000,3.000000
max,30899.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.985764,4.000000,4.000000,4.000000


In [35]:
attribute_df1 = attribute_df.merge(node_df, left_on=['Compno'], right_on=['ComputerNo']).drop(['ComputerNo'], axis=1)

KeyError: 'ComputerNo'

In [ ]:
attr = attribute_df[['TimeStamp', 'Compno']].sort_values(by='TimeStamp')
attr = attr.dropna(subset=['Compno'])

In [ ]:
# Rename all columns by adding '_x' suffix 
attr_x = attribute_df.add_suffix('_x')
attr_y = attribute_df.add_suffix('_y')

In [ ]:
# Perform cross join within each TimeStamp group 
cross_joined = attr.groupby('TimeStamp').apply(lambda x: x.assign(key=1).merge(x.assign(key=1), on='key')).drop('key', axis=1).reset_index(drop=True)
cross_joined[['TimeStamp']] = cross_joined[['TimeStamp_x']]
correct_attr = cross_joined.drop(['TimeStamp_x', 'TimeStamp_y'], axis=1)[['TimeStamp', 'Compno_x', 'Compno_y']]

In [ ]:
# Merge DataFrames on 'Key1_df1' and 'Key2_df1' from df1 with 'Key1_df2' and 'Key2_df2' from df2 
merged_df = correct_attr.merge(attr_x, left_on=['TimeStamp', 'Compno_x'], right_on=['TimeStamp_x', 'Compno_x']).drop(['TimeStamp_x', 'Year_x', 'Month_x'], axis=1)
final_df_1 = merged_df.merge(attr_y, left_on=['TimeStamp', 'Compno_y'], right_on=['TimeStamp_y', 'Compno_y']).drop(['TimeStamp_y', 'Year_y', 'Month_y'], axis=1)

In [ ]:
final_df = final_df_1.merge(edges_df, left_on=['Compno_x', 'Compno_y'], right_on=['CompNo1', 'CompNo2']).drop(['CompNo1', 'CompNo2'], axis=1)

In [ ]:
if final_df['distance_km'].isnull().any():
    print("There are missing values in the 'distance_km' column.")
    final_df = final_df.dropna(subset=['distance_km'])


In [ ]:
# Load the dataset
df = final_df

# Initialize dictionary to store graphs by timestamp
graphs_by_timestamp = {}

# Process data into graphs
for timestamp in df["TimeStamp"].unique():
    # Filter rows for the current timestamp
    sub_df = df[df["TimeStamp"] == timestamp]

    # Create graph for the timestamp
    G = nx.Graph()
    for _, row in sub_df.iterrows():
        # Add nodes and features
        G.add_node(row["Compno_x"], features=row[["AverageMaximumTemperature_x", 
                                                 "AverageMinimunTemperature_x", 
                                                 "AverageMaximumRelativeHumidity_x", 
                                                 "AverageMinimumRelativeHumidity_x", 
                                                 "TotalRainfall_x", 
                                                 "AverageTotalRadiation_x", 
                                                 "AverageWindSpeed_x", 
                                                 "AverageTotalRelativeEvaporation_x"]].values)
        G.add_node(row["Compno_y"], features=row[["AverageMaximumTemperature_y", 
                                                 "AverageMinimunTemperature_y", 
                                                 "AverageMaximumRelativeHumidity_y", 
                                                 "AverageMinimumRelativeHumidity_y", 
                                                 "TotalRainfall_y", 
                                                 "AverageTotalRadiation_y", 
                                                 "AverageWindSpeed_y", 
                                                 "AverageTotalRelativeEvaporation_y"]].values)
        # Add edge with weight
        G.add_edge(row["Compno_x"], row["Compno_y"], weight=row["distance_km"])
    
    # Store graph
    graphs_by_timestamp[timestamp] = G

# Feature propagation for missing nodes
def propagate_features(graph):
    for node in graph.nodes:
        if "features" not in graph.nodes[node]:
            # Aggregate features from neighbors
            neighbor_features = [
                graph.nodes[neighbor].get("features")
                for neighbor in graph.neighbors(node)
                if "features" in graph.nodes[neighbor]
            ]
            if neighbor_features:
                graph.nodes[node]["features"] = np.mean(neighbor_features, axis=0)
            else:
                graph.nodes[node]["features"] = np.zeros(8)  # Default to zero vector

# Apply feature propagation to all graphs
#for timestamp, G in graphs_by_timestamp.items():
#    propagate_features(G)

In [ ]:
results_df = {}  # Initialize the results dictionary

for timestamp in df["TimeStamp"].unique():  # Iterate through unique timestamps
    new_df = graphs_by_timestamp[timestamp].nodes()  # Get nodes for the current timestamp
    results_df[timestamp] = new_df  # Add the data to the dictionary

In [ ]:
# The list of labels (all possible unique values across the dictionary)
labels = [30142, 30173, 30191, 30389, 30688, 30692, 30899]

# Build the dataframe
data = []
for timestamp, nodes in results_df.items():
    row = {"TimeStamp": timestamp}
    for label in labels:
        row[label] = label in nodes
    data.append(row)

df = pd.DataFrame(data)

In [ ]:
df = final_df

# Function to encode columns starting from 0 for each group
def encode_columns(group):
    le_x = LabelEncoder()
    le_y = LabelEncoder()
    group['Compno_x_encoded'] = le_x.fit_transform(group['Compno_x'])
    group['Compno_y_encoded'] = le_y.fit_transform(group['Compno_y'])
    return group

# Apply the encoding function to each group
df = df.groupby('TimeStamp').apply(encode_columns)

final_backup = final_df

final_df = df

final_df[['Compno_x', 'Compno_y']] = final_df[['Compno_x_encoded', 'Compno_y_encoded']]
final_df.drop(['Compno_x_encoded', 'Compno_y_encoded'], axis=1, inplace=True)

In [ ]:
final_df = final_backup

In [ ]:
# Convert 'Timestamp' column to datetime (if required)
df["TimeStamp"] = pd.to_datetime(df["TimeStamp"])

# Count the number of 0 values in each row
rows_with_two_or_more_zeros = df[df.eq(False).sum(axis=1) > 2]

# Select the timestamps
timestamps = rows_with_two_or_more_zeros["TimeStamp"]

print('Number of Timestamps: ' , df['TimeStamp'].nunique())
print('Number of Timestamps: ' , len(timestamps))
print('Remaining Graphs: ' , (int(df['TimeStamp'].nunique()) - int(len(timestamps))))
#print(timestamps)

In [ ]:
# Drop rows where 'Timestamp' is in the list
df = df[~df["TimeStamp"].isin(timestamps)]

# Building the Model

In [36]:
# Drop rows with missing values (if any)
analysis_df = analysis_df.dropna()

# Separate features (X) and target (y)
X = analysis_df.drop(columns=['Target', 'TimeStamp', 'Compno'])
y = analysis_df['Target']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
# Define the parameter grid
param_grid = {
    'n_estimators': [10, 25, 50, 100],  # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'max_features': ['auto', 'sqrt', 'log2']  # Number of features to consider for best split
}

# Initialize the Random Forest model
rf = RandomForestClassifier(random_state=42)

# Perform GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best parameters and the best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Score:", best_score)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
Best Parameters: {'max_depth': None, 'max_features': 'auto', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}
Best Score: 0.7979899497487437


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\ensemble\_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


In [38]:
# Use the best model from GridSearchCV
best_model = grid_search.best_estimator_

# Make predictions on the test set
y_pred_best = best_model.predict(X_test)

# Evaluate the model
print("Accuracy with Best Model:", accuracy_score(y_test, y_pred_best))

# Generate and display the classification report
print("\nClassification Report with Best Model:")
print(classification_report(y_test, y_pred_best))


Accuracy with Best Model: 0.7791164658634538

Classification Report with Best Model:
              precision    recall  f1-score   support

         0.0       0.63      0.65      0.64        49
         1.0       0.88      0.69      0.77        42
         2.0       0.69      0.79      0.73        56
         3.0       0.86      0.82      0.84        51
         4.0       0.90      0.92      0.91        51

    accuracy                           0.78       249
   macro avg       0.79      0.77      0.78       249
weighted avg       0.79      0.78      0.78       249



In [41]:
# Save the model to a file
joblib.dump(best_model, 'RandomForestClassifier.pkl')

# Later, load the model from the file
# loaded_model = joblib.load('RandomForestClassifier.pkl')

['RandomForestClassifier.pkl']

In [42]:
# Define the function to create the neural network model
def create_model(learning_rate=0.001, activation='relu'):
    model = Sequential([
        # First hidden layer with more neurons
        Dense(128, activation=activation, input_shape=(X_train.shape[1],)),
        BatchNormalization(),  # Batch normalization added
        Dropout(0.3),  # Regularization with dropout
        
        # Second hidden layer
        Dense(64, activation=activation),
        BatchNormalization(),  # Batch normalization added
        Dropout(0.3),

        # Third hidden layer
        Dense(32, activation=activation),
        BatchNormalization(),  # Batch normalization added
        Dropout(0.3),
        
        # Forth hidden layer
        Dense(32, activation=activation),
        BatchNormalization(),  # Batch normalization added
        Dropout(0.3),

        # Output layer for classification
        Dense(len(y.unique()), activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Wrap the Keras model so it works with GridSearchCV
keras_clf = KerasClassifier(build_fn=create_model, verbose=0)

C:\Users\ZAMA\AppData\Local\Temp\ipykernel_28140\4292293066.py:31: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  keras_clf = KerasClassifier(build_fn=create_model, verbose=0)


In [45]:
# Define the grid of hyperparameters to search
param_grid = {
    'batch_size': [16, 32, 64],
    'epochs': [120, 135, 150],
    'learning_rate': [0.0025, 0.003, 0.0035,],
    'activation': ['relu', 'tanh', 'swish']
}

# Perform the grid search
grid = GridSearchCV(estimator=keras_clf, param_grid=param_grid, cv=3, verbose=2, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters and the best accuracy score
print(f"Best Parameters: {grid_result.best_params_}")
print(f"Best Score: {grid_result.best_score_}")

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Best Parameters: {'activation': 'swish', 'batch_size': 32, 'epochs': 135, 'learning_rate': 0.0025}
Best Score: 0.7668256163597107


In [46]:
# Use the best model from GridSearchCV
best_model = grid_result.best_estimator_

# Make predictions on the test set
y_pred_best = best_model.predict(X_test)

# Evaluate accuracy
test_accuracy = accuracy_score(y_test, y_pred_best)
print(f"Test Accuracy with Best Model: {test_accuracy:.4f}")

# Generate and display the classification report
print("\nClassification Report with Best Model:")
print(classification_report(y_test, y_pred_best))


8/8 [==============================] - 1s 4ms/step
Test Accuracy with Best Model: 0.7751

Classification Report with Best Model:
              precision    recall  f1-score   support

         0.0       0.76      0.69      0.72        49
         1.0       0.67      0.62      0.64        42
         2.0       0.70      0.80      0.75        56
         3.0       0.90      0.88      0.89        51
         4.0       0.84      0.84      0.84        51

    accuracy                           0.78       249
   macro avg       0.77      0.77      0.77       249
weighted avg       0.78      0.78      0.77       249



In [47]:
# Save the entire model to a file (including architecture, optimizer, and weights)
best_model.model.save('GridSearchCV.h5')

# Later, you can load the model back using:
#best_model = load_model('RandomForestClassifier.h5')

In [ ]:
# Assuming 'final_df' has a 'timestamp' column
timestamps = final_df['TimeStamp'].unique()

# Initialize lists to store data for each timestamp
edge_indices_list = []
edge_weights_list = []
node_features_list = []
targets_list = []
all_batches = []

for timestamp in timestamps:
    df_timestamp = final_df[final_df['TimeStamp'] == timestamp]
    
    # Node features
    node_features = df_timestamp[['Compno_x', 'AverageMaximumTemperature_x', 'AverageMinimunTemperature_x', 
                                  'AverageMaximumRelativeHumidity_x', 'AverageMinimumRelativeHumidity_x', 
                                  'TotalRainfall_x', 'AverageTotalRadiation_x', 'AverageWindSpeed_x', 
                                  'AverageTotalRelativeEvaporation_x']].drop_duplicates()
    node_features = node_features.sort_values(by='Compno_x')
    x = torch.tensor(node_features[['AverageMaximumTemperature_x', 'AverageMinimunTemperature_x', 
                                    'AverageMaximumRelativeHumidity_x', 'AverageMinimumRelativeHumidity_x', 
                                    'TotalRainfall_x', 'AverageTotalRadiation_x', 'AverageWindSpeed_x', 
                                    'AverageTotalRelativeEvaporation_x']].values, dtype=torch.float)
    
    # Edge indices
    edge_indices = torch.tensor(df_timestamp[['Compno_x', 'Compno_y']].values.T, dtype=torch.long)
    
    # Edge weights
    edge_weights = torch.tensor(df_timestamp['distance_km'].values, dtype=torch.float)
    
    #edge_indices = [torch.randint(0, n_nodes, (2, n_edges), dtype=torch.long) for _ in df_timestamps] 
    #edge_weights = [torch.rand(n_edges) for _ in timestamps]
    
    # Targets
    targets = df_timestamp[['Compno_x', 'Target_x']].drop_duplicates()['Target_x'].values
    targets_np = np.array(targets, dtype=np.float32)
    targets_tensor = torch.tensor(targets_np, dtype=torch.float)
    
    # Generate batches (assuming each timestamp is a separate batch)
    batch = torch.tensor([i for i in range(len(df_timestamp))], dtype=torch.long)
    
    # Append to lists
    node_features_list.append(x)
    edge_indices_list.append(edge_indices)
    edge_weights_list.append(edge_weights)
    targets_list.append(targets_tensor)
    all_batches.append(batch)

# Ensure the length consistency across all lists
assert len(edge_indices_list) == len(edge_weights_list) == len(node_features_list) == len(targets_list) == len(all_batches), "Temporal dimension inconsistency."


In [ ]:
# Create a DynamicGraphTemporalSignal object
dataset = DynamicGraphTemporalSignalBatch(
    edge_indices=edge_indices_list,
    edge_weights=edge_weights_list,
    features=node_features_list,
    targets=targets_list,
    batches=all_batches
)

In [ ]:
# Example usage
i = 15
print("DynamicGraphTemporalSignal created with the following properties:")
print(f"Number of snapshots\t: {dataset.snapshot_count}")
print(f"Snapshot {i} Features\t: \n{dataset.features[i]}")
print(f"Snapshot {i} Edges\t: \n{dataset.edge_indices[i]}")
print(f"Snapshot {i} Edge Weights\t: \n{dataset.edge_weights[i]}")
print(f"Snapshot {i} Targets\t\t: \n{dataset.targets[i]}")

In [ ]:
# Assuming 'dynamic_graph_temporal_signal_batch' is an instance of DynamicGraphTemporalSignalBatch

def get_individual_snapshots(dynamic_graph_temporal_signal_batch):
    snapshots = []
    for time, snapshot in enumerate(dynamic_graph_temporal_signal_batch):
        snapshots.append(snapshot)
        '''
        print(f"--- Snapshot {time} ---")
        print(f"Edge Indices:\n{snapshot.edge_index}")
        print(f"Edge Weights:\n{snapshot.edge_attr}")
        print(f"Node Features:\n{snapshot.x}")
        print(f"Targets:\n{snapshot.y}")
        print("\n")
        '''
    return snapshots

# Example usage
snapshots = get_individual_snapshots(dataset)


In [ ]:
model.eval()
predictions, targets = [], []
with torch.no_grad():
    for snapshot in test_loader:
        snapshot = snapshot.to(device)
        out = model(snapshot.x, snapshot.edge_index, snapshot.edge_weight)
        predictions.append(out.cpu().numpy())
        targets.append(snapshot.y.cpu().numpy())

# Convert lists to arrays for further evaluation
predictions = np.array(predictions)
targets = np.array(targets)

# Evaluate model performance (e.g., using RMSE, MAE, etc.)
rmse = np.sqrt(np.mean((predictions - targets) ** 2))
print(f'RMSE: {rmse}')
